# Lineage and Memory that change decisions · 2.1.5

**Input → process → output:** payment PAY-001 arrives with 5000 MXN cents. One Agent computes a fee of 50; another checks its evidence and decides. The first run approves. The second receives exactly the same input and requests duplicate review because Memory retained the previous episode.

Everything uses deterministic Python coordinated by LangGraph: no models, keys or inference. Install local SDK **2.1.4** and its dependencies first. The notebook installs nothing. Run cells in order from a clean kernel.

| Channel | Contents | Decision role |
|---|---|---|
| State | payment_id, amount, fee, decision | Current payment data |
| Context.data / sources | Policy and provenance | Fee and limit |
| Context.lineage | Preceding Agent evidence | Require and verify computation |
| Context.episodes | Prior execution with compact lineage | Detect prior approval of the same payment |

Context captures are separate inspection records, never Graph state. Lineage records observations, not private thoughts.

In [2]:
import pydantic
import agentic_systems as toolkit

assert toolkit.__version__ == "2.1.5"
print("SDK:", toolkit.__version__)

SDK: 2.1.4.dev8


## 1. Small state and explicit Tools

strict=True rejects type coercion; extra="forbid" rejects unexpected fields. Optional fields are outputs not yet computed. The reviewer does not trust a fee solely because it appears in state: successful fees lineage and consistency with current policy are required. It then checks structured evidence of previous approvals in Memory.

In [3]:
class Payment(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(strict=True, extra="forbid")
    payment_id: str = pydantic.Field(min_length=1)
    amount: int = pydantic.Field(gt=0)
    fee: int | None = None
    decision: str | None = None

In [4]:
# Observation captures are separate from state.
received_contexts = []

@toolkit.tool
def calculate_fee(amount: int, *, context: toolkit.Context) -> dict:
    received_contexts.append({"agent": "fees", "context": context.to_dict()})
    return {"fee": amount * context.data["fee_bps"] // 10000}


@toolkit.tool
def decide(payment_id: str, amount: int, fee: int, *, context: toolkit.Context) -> dict:
    received_contexts.append({"agent": "reviewer", "context": context.to_dict()})

    # Current evidence must support the fee before this agent may approve it.
    fee_evidence = [row for row in context.lineage if row["name"] == "fees"]
    if not fee_evidence or not fee_evidence[-1]["ok"]:
        raise ValueError("Missing successful fees lineage")
    expected_fee = amount * context.data["fee_bps"] // 10000
    if fee_evidence[-1]["data"].get("fee") != fee or fee != expected_fee:
        raise ValueError("Fee is inconsistent with lineage or current policy")

    # Previous approval of the same business identifier changes the decision.
    # Use structured recorded evidence, never parse a human-readable explanation.
    for episode in context.episodes:
        if not episode["ok"]:
            continue
        for step in episode["lineage"]["steps"]:
            if step["kind"] != "execution" or step["title"] != "payments":
                continue
            evidence = step["evidence"]
            previous = evidence.get("data", {})
            if (evidence["ok"] and previous.get("payment_id") == payment_id
                    and previous.get("decision") == "approve"):
                return {"decision": "review_duplicate"}

    return {
        "decision": "approve" if amount + fee <= context.data["ceiling"] else "review"
    }

## 2. One Graph per Agent and one for the System

fees organizes computation; reviewer organizes the decision. The System connects them in that order. There is no manual global-trace lookup or history attached to Payment. Context is delivered when each Tool executes.

In [5]:
fee_graph =toolkit.Graph(Payment)
fee_graph.add_node("calculate_fee", calculate_fee)
fee_graph.add_edge(toolkit.START, "calculate_fee")
fee_graph.add_edge("calculate_fee", toolkit.END)
fee_agent = toolkit.Agent("fees", fee_graph)

review_graph = toolkit.Graph(Payment)
review_graph.add_node("decide", decide)
review_graph.add_edge(toolkit.START, "decide")
review_graph.add_edge("decide", toolkit.END)
reviewer = toolkit.Agent("reviewer", review_graph)

payment_graph = toolkit.Graph(Payment)
payment_graph.add_node("fees", fee_agent)
payment_graph.add_node("reviewer", reviewer)
payment_graph.add_edge(toolkit.START, "fees")
payment_graph.add_edge("fees", "reviewer")
payment_graph.add_edge("reviewer", toolkit.END)

## 3. Policy and shared Memory from the System

Memory retains up to two local episodes. Entry snapshots of prior episodes are inherited by Agents. The System remembers its result on completion. compile() prepares the harness without executing Tools or writing Memory.

In [6]:
memory =toolkit.Memory(capacity=2)
system = toolkit.System(
    "payments",
    payment_graph,
    context=toolkit.Context(
        {"currency": "MXN", "fee_bps": 100, "ceiling": 10000},
        sources=("payment-policy-v1",),
    ),
    memory=memory,
)

harness = system.compile()
assert received_contexts == [] and memory.read() == []

## 4. First execution: current evidence, no history yet

fees receives empty lineage and episodes. reviewer receives current fees evidence but no previous episode, so PAY-001 can be approved.

In [7]:
payment = {"payment_id": "PAY-001", "amount": 5000}
first = harness.run(payment)
assert first.ok
assert first.data == {**payment, "fee": 50, "decision": "approve"}
first_contexts = list(received_contexts)
assert len(first_contexts) == 2
assert len(memory.read()) == 1
print(first.human_results())

payments: COMPLETED
Execution: run-10043c15b7ac4ed3bd549d001b112e61
Input: {"payment_id":"PAY-001","amount":5000}
Output: {"payment_id":"PAY-001","amount":5000,"fee":50,"decision":"approve"}
Process:
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}


### Exact Context received in the first execution

The complete object is shown: policy, sources, current lineage and episodes.

In [8]:
import json

for row in first_contexts:
    print("Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [],
  "lineage": []
}
Agent: reviewer
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [],
  "lineage": [
    {
      "data": {
        "fee": 50
      },
      "execution_id": "run-46b56f4ed84e4d88b774996a8aeeb6db",
      "name": "fees",
      "ok": true
    }
  ]
}


## 5. Second execution: identical input changes the decision through Memory

The same harness and Memory are reused. Fee and limit remain unchanged. reviewer finds previous approval of PAY-001 in episode lineage and returns review_duplicate. Execution succeeds: requesting review is a business decision, not a runtime failure.

In [9]:
second = harness.run(payment)
assert second.ok
assert second.data == {**payment, "fee": 50, "decision": "review_duplicate"}
second_contexts = list(received_contexts[2:])
assert len(second_contexts) == 2
assert len(memory.read()) == 2
print(second.human_results())

payments: COMPLETED
Execution: run-3bfadcdc65084eba92fb8075ac993539
Input: {"payment_id":"PAY-001","amount":5000}
Output: {"payment_id":"PAY-001","amount":5000,"fee":50,"decision":"review_duplicate"}
Process:
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"review_duplicate"}


### Exact Context received in the second execution

Both Tools receive the first episode in episodes. Only reviewer also receives current fees evidence in lineage. The first Agent does not confuse previous-run history with work already performed in this run.

In [10]:
for row in second_contexts:
    print("Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [
    {
      "context": {
        "episode_ids": [],
        "fingerprint": "2b78cb800cc7bca055249311d20ea1121ffd1004657a1ef7a0c2c58c7d8ddaad",
        "memory_revision": 0,
        "sources": [
          "payment-policy-v1"
        ]
      },
      "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
      "lineage": {
        "answer": "{\"payment_id\": \"PAY-001\", \"amount\": 5000, \"fee\": 50, \"decision\": \"approve\"}",
        "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
        "failed_steps": 0,
        "ok": true,
        "omitted_steps": 0,
        "steps": [
          {
            "evidence": {
              "data": {
                "amount": 5000,
                "decision": "approve",
                "fee": 50,
                "payment_id": "PAY-001"
              },
              "errors": [],
         

## 6. Controls demonstrating causal dependence

1. Same input and Graph without Memory: approve again.
2. fee=50 in state without preceding fees lineage: fail.
3. fees records fee=51: lineage exists but contradicts policy, so fail.

Controls use independent Systems and do not write this example’s Memory.

In [11]:
without_memory = toolkit.System(
    "payments", payment_graph, context=system.context,
).run(payment)
assert without_memory.ok and without_memory.data["decision"] == "approve"
print("Without Memory:")
print(without_memory.human_results())

missing_lineage = toolkit.System(
    "missing_evidence", review_graph, context=system.context,
).run({**payment, "fee": 50})
assert not missing_lineage.ok
assert "Missing successful fees lineage" in str(missing_lineage.errors)
print("\nWithout prior lineage:")
print(missing_lineage.human_results())

Without Memory:
payments: COMPLETED
Execution: run-b80204c715154a9bb7afbe5814bae4f4
Input: {"payment_id":"PAY-001","amount":5000}
Output: {"payment_id":"PAY-001","amount":5000,"fee":50,"decision":"approve"}
Process:
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}

Without prior lineage:
missing_evidence: FAILED
Execution: run-e7218c417e1e4a6c8257db8a63caf58b
Input: {"payment_id":"PAY-001","amount":5000,"fee":50}
Output: {}
Errors: [{"code":"_NodeFailure","message":"9 validation errors for Payment\ncategory\n  Extra inputs are not permitted [type=extra_forbidden, input_value='provider', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden\nmessage\n  Extra inputs are not permitted [type=extra_forbidden, input_value='Missing successful fees lineage', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/extra

In [12]:
def incorrect_fee(state):
    return {"fee": 51}


bad_graph = toolkit.Graph(Payment)
bad_graph.add_node("fees", incorrect_fee)
bad_graph.add_node("reviewer", reviewer)
bad_graph.add_edge(toolkit.START, "fees")
bad_graph.add_edge("fees", "reviewer")
bad_graph.add_edge("reviewer", toolkit.END)

inconsistent_lineage = toolkit.System(
    "inconsistent_evidence", bad_graph, context=system.context,
).run(payment)
assert not inconsistent_lineage.ok
assert "Fee is inconsistent" in str(inconsistent_lineage.errors)
assert len(memory.read()) == 2
print(inconsistent_lineage.human_results())

inconsistent_evidence: FAILED
Execution: run-3d5bdf410af246b688cbdbb820c9757b
Input: {"payment_id":"PAY-001","amount":5000}
Output: {}
Errors: [{"code":"_NodeFailure","message":"Required node failed: reviewer"},{"code":"_NodeFailure","message":"9 validation errors for Payment\ncategory\n  Extra inputs are not permitted [type=extra_forbidden, input_value='provider', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden\nmessage\n  Extra inputs are not permitted [type=extra_forbidden, input_value='Fee is inconsistent with...neage or current policy', input_type=str]\n    For further informa…
Process:
- Step fees: completed
- Agent reviewer: failed
- Tool decide: failed -> {"category":"provider","message":"Fee is inconsistent with lineage or current policy","provider":"python-runtime","framework":"native","code":"ValueError","retryable":false,"cause_type":"ValueError","details":{},"error_type":"ValueError"}


## 7. Verified summary: four Context objects, four business fields

In [13]:
main_contexts = first_contexts + second_contexts
assert len(main_contexts) == 4
print("Run | Agent    | current lineage | previous episodes")
for index, row in enumerate(main_contexts):
    context = row["context"]
    assert context["data"] == {"currency": "MXN", "fee_bps": 100, "ceiling": 10000}
    assert context["sources"] == ["payment-policy-v1"]
    assert len(context["episodes"]) == index // 2
    if row["agent"] == "fees":
        assert context["lineage"] == []
    else:
        assert any(item["name"] == "fees" and item["ok"] for item in context["lineage"])
    if index >= 2:
        assert context["episodes"][0]["execution_id"] == first.execution_id
    print(index // 2 + 1, row["agent"], len(context["lineage"]), len(context["episodes"]), sep=" | ")

assert set(first.data) == set(second.data) == {"payment_id", "amount", "fee", "decision"}
print("\nFirst result:", first.data)
print("Second result:", second.data)
print("State fields:", list(Payment.model_fields))
print("Retained episodes:", len(memory.read()))
print("Two executions and three controls: passed.")

Run | Agent    | current lineage | previous episodes
1 | fees | 0 | 0
1 | reviewer | 1 | 0
2 | fees | 0 | 1
2 | reviewer | 1 | 1

First result: {'payment_id': 'PAY-001', 'amount': 5000, 'fee': 50, 'decision': 'approve'}
Second result: {'payment_id': 'PAY-001', 'amount': 5000, 'fee': 50, 'decision': 'review_duplicate'}
State fields: ['payment_id', 'amount', 'fee', 'decision']
Retained episodes: 2
Two executions and three controls: passed.


## 8. Environment: Memory evolves between episodes

Process PAY-002 twice through the same System. observe supplies only payment fields; expected_decision stays outside System input as an Environment oracle. update records an explicit reward and captures Memory separately. Capacity is two: episode count stops growing but IDs change. No additional Memory is attached to Environment; we observe the System’s Memory.

In [14]:
environment_memory_history = []


def observe_payment(record, state):
    return {"payment_id": record["payment_id"], "amount": record["amount"]}


def update_episode(state, record, result):
    environment_memory_history.append({
        "payment_id": record["payment_id"],
        "decision": result.data["decision"],
        "memory": memory.read(),
    })
    reward = int(result.ok and result.data["decision"] == record["expected_decision"])
    return {"processed": state["processed"] + 1,
            "reward": reward, "total_reward": state["total_reward"] + reward}


episode_graph = toolkit.Graph(Payment)
episode_graph.add_node("payment", system)
episode_graph.add_edge(toolkit.START, "payment")
episode_graph.add_edge("payment", toolkit.END)

environment = toolkit.Environment(
    "payment_day", graph=episode_graph,
    records=(
        {"payment_id": "PAY-002", "amount": 5000, "expected_decision": "approve"},
        {"payment_id": "PAY-002", "amount": 5000, "expected_decision": "review_duplicate"},
    ),
    state={"processed": 0, "reward": 0, "total_reward": 0},
    observe=observe_payment, update=update_episode,
)

In [15]:
memory_before_environment = memory.read()
environment_context_start = len(received_contexts)
episode = environment.run(timeout_s=30)
assert episode.ok
assert [row["decision"] for row in episode.data["outputs"]] == ["approve", "review_duplicate"]
assert episode.data["summary"]["state"]["total_reward"] == 2
environment_contexts = list(received_contexts[environment_context_start:])
assert len(environment_contexts) == 4 and len(environment_memory_history) == 2
assert all(len(row["memory"]) == 2 for row in environment_memory_history)
assert environment_memory_history[0]["memory"][0]["execution_id"] == second.execution_id
assert environment_memory_history[1]["memory"][0]["execution_id"] == environment_memory_history[0]["memory"][1]["execution_id"]
assert [e["execution_id"] for e in environment_contexts[2]["context"]["episodes"]] == [e["execution_id"] for e in environment_memory_history[0]["memory"]]
print(episode.human_results())
print("\nEnvironment state:", episode.data["summary"]["state"])
print("Before:", [e["execution_id"] for e in memory_before_environment])
for index, row in enumerate(environment_memory_history, 1):
    print("After episode", index, row["decision"], [e["execution_id"] for e in row["memory"]])

payment_day: COMPLETED
Execution: run-caa00e0704bb46949cc7ab123bcb1b19
Input: {"records_observed":[{"payment_id":"PAY-002","amount":5000,"expected_decision":"approve"},{"payment_id":"PAY-002","amount":5000,"expected_decision":"review_duplicate"}]}
Output: {"outputs":[{"payment_id":"PAY-002","amount":5000,"fee":50,"decision":"approve"},{"payment_id":"PAY-002","amount":5000,"fee":50,"decision":"review_duplicate"}],"summary":{"state":{"processed":2,"reward":1,"total_reward":2},"processed":2,"terminated":true,"truncated":false,"reason":"records_exhausted"}}
Process:
- System payments: completed
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}
- System payments: completed
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
2 operations omitted; inspect lineage for full evidence.

Environment state: {'processed': 2, 'reward': 1, 'total_reward': 2}
Before: ['run-10043c15b7ac4e

### Exact Context objects during Environment

The second Tool pair receives the first PAY-002 episode. The observation list is not added to state.

In [16]:
for index, row in enumerate(environment_contexts):
    print("Episode", index // 2 + 1, "Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

Episode 1 Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [
    {
      "context": {
        "episode_ids": [],
        "fingerprint": "2b78cb800cc7bca055249311d20ea1121ffd1004657a1ef7a0c2c58c7d8ddaad",
        "memory_revision": 0,
        "sources": [
          "payment-policy-v1"
        ]
      },
      "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
      "lineage": {
        "answer": "{\"payment_id\": \"PAY-001\", \"amount\": 5000, \"fee\": 50, \"decision\": \"approve\"}",
        "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
        "failed_steps": 0,
        "ok": true,
        "omitted_steps": 0,
        "steps": [
          {
            "evidence": {
              "data": {
                "amount": 5000,
                "decision": "approve",
                "fee": 50,
                "payment_id": "PAY-001"
              },
              "errors": [],

## 9. Eval: live episodes within a case, isolated Memory between cases

Evaluate the same Environment twice, each case repeating PAY-003. Each starts with the same production Memory containing PAY-002. Within each case, first approval of PAY-003 feeds the second episode and changes its decision. The next case returns to production baseline and approves its first PAY-003.

The check reads Memory captured by closure and also receives the private case view. After Eval, exact equality of production Memory is required. Audit lists still accumulate observations: Eval does not isolate files, arbitrary variables, SDK clients or external effects.

In [17]:
eval_memory_views = []


def check_episode(result, case):
    decisions = [row["decision"] for row in result.data["outputs"]]
    eval_memory_views.append({"case": case["name"], "memory": memory.read()})
    return {"ok": decisions == ["approve", "review_duplicate"]
            and result.data["summary"]["state"]["total_reward"] == 2}


eval_records = (
    {"payment_id": "PAY-003", "amount": 5000, "expected_decision": "approve"},
    {"payment_id": "PAY-003", "amount": 5000, "expected_decision": "review_duplicate"},
)
evaluation = toolkit.Eval(
    "isolated_payment_days", unit=environment,
    cases=(
        {"name": "case-a", "input": {"records": eval_records}, "expected": {"ok": True}},
        {"name": "case-b", "input": {"records": eval_records}, "expected": {"ok": True}},
    ),
    checks=(check_episode,),
)

In [18]:
production_before_eval = memory.read()
production_ids = [e["execution_id"] for e in production_before_eval]
eval_context_start = len(received_contexts)
evaluated = evaluation.run(timeout_s=30)
eval_contexts = list(received_contexts[eval_context_start:])
assert evaluated.ok and evaluated.data["passed"] == 2
assert evaluated.meta["memory_isolation"] == "per_case"
assert memory.read() == production_before_eval
assert len(eval_contexts) == 8 and len(eval_memory_views) == 2
for offset in (0, 4):
    first_episode = eval_contexts[offset]["context"]["episodes"]
    second_episode = eval_contexts[offset + 2]["context"]["episodes"]
    assert [e["execution_id"] for e in first_episode] == production_ids
    assert [e["execution_id"] for e in second_episode] != production_ids
assert [e["execution_id"] for e in eval_memory_views[0]["memory"]] != [e["execution_id"] for e in eval_memory_views[1]["memory"]]
print(evaluated.human_results())
print("\nProduction Memory before:", production_ids)
for row in eval_memory_views:
    print("Private view", row["case"], [e["execution_id"] for e in row["memory"]])
print("Production Memory after:", [e["execution_id"] for e in memory.read()])
print("Exact equality before/after:", memory.read() == production_before_eval)

isolated_payment_days: COMPLETED
Execution: run-ee73b0dc43404921833d2506062bf4de
Input: {"cases_executed":[{"records":[{"payment_id":"PAY-003","amount":5000,"expected_decision":"approve"},{"payment_id":"PAY-003","amount":5000,"expected_decision":"review_duplicate"}]},{"records":[{"payment_id":"PAY-003","amount":5000,"expected_decision":"approve"},{"payment_id":"PAY-003","amount":5000,"expected_decision":"review_duplicate"}]}]}
Output: {"total":2,"passed":2,"failed":0}
Process:
- Environment payment_day: completed
- System payments: completed
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}
- System payments: completed
- Agent fees: completed
14 operations omitted; inspect lineage for full evidence.

Production Memory before: ['run-9956139e296e4b17ab860471a1729c2c', 'run-bc6f52d4511545d3b632dc4f14753b83']
Private view case-a ['run-65acfe1e2794468a8ae0c9b3193c63d1', 'run-8044b9c498b74ae187

### Exact Context objects during Eval

Eight Context objects are shown: two cases × two episodes × two Agents. Both cases start with the same production IDs; writes from one case do not reach the other.

In [19]:
for index, row in enumerate(eval_contexts):
    print("Case", index // 4 + 1, "episode", (index % 4) // 2 + 1, "Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

for result in (first, second, episode, evaluated):
    assert result.check_invariants().ok
    assert result.usage.get("inference_requests", 0) == 0
assert set(Payment.model_fields) == {"payment_id", "amount", "fee", "decision"}
print("Environment and Eval passed; production Memory unchanged after Eval.")

Case 1 episode 1 Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [
    {
      "context": {
        "episode_ids": [
          "run-10043c15b7ac4ed3bd549d001b112e61",
          "run-3bfadcdc65084eba92fb8075ac993539"
        ],
        "fingerprint": "455e5946dbc0274969ce803033c3858d97d35144b265b6675c62bc42f8c551fa",
        "memory_revision": 2,
        "sources": [
          "payment-policy-v1"
        ]
      },
      "execution_id": "run-9956139e296e4b17ab860471a1729c2c",
      "lineage": {
        "answer": "{\"payment_id\": \"PAY-002\", \"amount\": 5000, \"fee\": 50, \"decision\": \"approve\"}",
        "execution_id": "run-9956139e296e4b17ab860471a1729c2c",
        "failed_steps": 0,
        "ok": true,
        "omitted_steps": 0,
        "steps": [
          {
            "evidence": {
              "data": {
                "amount": 5000,
                "decision": "approv

## Scope and limits

Agent/System lineage can provide context to another unit. Context may also contain policy and external sources; lineage alone is only a projection. We inspect complete objects, but decisions read structured fields rather than human_results text.

Memory is local, bounded and summarized. Absence of an approval in retained episodes does **not prove** it never happened. This demonstration is not a persistent idempotency ledger and does not prevent concurrent duplicates. The four business fields contain no Context, episodes or RunResult trees; internally the graph uses bounded evidence references.

Validation covers this deterministic scenario and its controls. It does not certify external providers or new versions and publishes no artifact.

[Base two-Agent program](example.py); Environment and Eval are declared directly in this notebook. [Explanation and evidence](README.md).